# General GPT-2 Attention Analysis

GPT-2 small analysis corresponding to Sections 3 and 6 of [What Does BERT Look At?](https://arxiv.org/abs/1906.04341).

Each prepared sample is `<|endoftext|> paragraph_1 \n\n paragraph_2 <|endoftext|>`. The two paragraphs are naturally adjacent in one Wikipedia article; no EOT is inserted at the paragraph boundary. GPT-2 attention is causal, so only keys at or before a query position are visible.

Experiment 2 reports attention-sink / potential no-op **candidates** from attention patterns alone. It does not establish functional no-op behavior. Experiment 3 reports both raw entropy and entropy normalized by the size of each causal support. All captions use the actual number of shards admitted by the 2 GiB dump budget.


In [1]:
import collections
import csv
import json
import os
import pickle

import matplotlib
import numpy as np
import seaborn as sns
import sklearn

from matplotlib import pyplot as plt
from matplotlib import cm
from sklearn import manifold

from gpt2_analysis import (
    compute_entropy_statistics, compute_token_sink_statistics,
    iter_attention_shards)

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.labelsize": 10,
    "legend.fontsize": 8.5,
    "legend.frameon": False,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.alpha": 0.15,
})

### Loading the Data
This notebook reads the budgeted GPT-2 attention shards produced by `extract_attention_gpt2.py`. The manifest records the actual analyzed sample count; it may be much smaller than the 1000 preprocessing candidates because dense attention is quadratic in sequence length.

In [2]:
ATTENTION_DIR = "./outputs/gpt2_attention"
RESULTS_DIR = "./results/gpt2"
FIGURE_DIR = os.path.join(RESULTS_DIR, "figures")
os.makedirs(FIGURE_DIR, exist_ok=True)

with open(os.path.join(ATTENTION_DIR, "manifest.json"), encoding="utf-8") as f:
  manifest = json.load(f)
n_docs = manifest["processed_count"]
if n_docs == 0:
  raise ValueError("manifest contains no attention shards")
print("Analyzing {} of {} candidates ({:.2f} MiB, stop reason: {})".format(
    n_docs, manifest["candidate_count"],
    manifest["total_output_bytes"] / (1024.0 ** 2),
    manifest["stop_reason"]))
print("Sequence lengths:", [s["sequence_length"] for s in manifest["shards"]])

first_metadata, first_attns = next(iter_attention_shards(ATTENTION_DIR))
n_layers, n_heads = first_attns.shape[:2]

def data_iterator():
  for metadata, attns in iter_attention_shards(ATTENTION_DIR):
    yield metadata["tokens"], np.asarray(attns)

def load_example(index=0):
  for i, (metadata, attns) in enumerate(iter_attention_shards(ATTENTION_DIR)):
    if i == index:
      return {"tokens": list(metadata["tokens"]), "attns": attns}
  raise IndexError(index)

def save_figure(name):
  for extension in ("png", "pdf"):
    plt.savefig(os.path.join(FIGURE_DIR, name + "." + extension),
                bbox_inches="tight", dpi=300)

sequence_lengths = [s["sequence_length"] for s in manifest["shards"]]
plt.figure(figsize=(7, 4))
plt.hist(sequence_lengths, bins=min(20, max(1, n_docs)), color="steelblue")
plt.xlabel("Sequence length (tokens)")
plt.ylabel("Documents")
plt.title("GPT-2 attention-shard length distribution (N={})".format(n_docs))
save_figure("input_length_distribution")
plt.show()

Analyzing 158 of 1000 candidates (2044.40 MiB, stop reason: storage_budget)
Sequence lengths: [119, 159, 309, 288, 133, 217, 155, 164, 158, 355, 328, 243, 255, 291, 629, 249, 201, 135, 403, 175, 475, 283, 151, 161, 289, 86, 132, 85, 329, 81, 214, 204, 295, 120, 239, 213, 73, 179, 164, 328, 170, 88, 98, 202, 218, 104, 102, 118, 91, 114, 299, 259, 194, 175, 355, 227, 102, 218, 266, 201, 404, 277, 185, 145, 146, 152, 168, 303, 114, 89, 126, 247, 134, 276, 98, 139, 209, 192, 101, 98, 227, 172, 94, 128, 194, 133, 223, 115, 155, 225, 216, 343, 215, 130, 135, 227, 128, 166, 195, 182, 138, 164, 231, 99, 85, 94, 367, 140, 234, 311, 145, 149, 192, 149, 131, 674, 120, 85, 115, 90, 272, 259, 145, 81, 286, 151, 156, 138, 338, 170, 93, 174, 91, 128, 123, 262, 139, 96, 363, 148, 195, 325, 159, 206, 142, 299, 167, 273, 166, 87, 242, 295, 108, 271, 101, 133, 106, 248]


/tmp/ipykernel_34050/3051864254.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Experiment 1: Relative Position and Legacy Token Categories
This section is retained for compatibility with the original analysis. Experiment 2 uses the frequency- and distance-adjusted token scan below.

In [3]:
avg_attns = {key: np.zeros((n_layers, n_heads)) for key in [
    "self", "right", "left", "sep", "sep_sep",
    "rest_sep", "cls", "punct"]}

print("Computing relative-position and legacy token stats")
for tokens, attns in data_iterator():
  n_tokens = attns.shape[-1]
  positions = np.arange(n_tokens)
  avg_attns["self"] += attns[:, :, positions, positions].mean(-1) / n_docs
  if n_tokens > 1:
    # right=previous and left=next preserve the labels in the original plot.
    avg_attns["right"] += attns[:, :, positions[1:], positions[:-1]].mean(-1) / n_docs
    avg_attns["left"] += attns[:, :, positions[:-1], positions[1:]].mean(-1) / n_docs
  avg_attns["cls"] += attns[:, :, :, 0].mean(-1) / n_docs

  sep_positions = [i for i, token in enumerate(tokens)
                   if token == "<|endoftext|>" and i != 0]
  punct_positions = [i for i, token in enumerate(tokens)
                     if token in (".", ",")]
  if sep_positions:
    avg_attns["sep"] += attns[:, :, :, sep_positions].sum(-1).mean(-1) / n_docs
    sep_queries = np.asarray(sep_positions)
    avg_attns["sep_sep"] += attns[:, :, sep_queries][:, :, :, sep_positions].sum(-1).mean(-1) / n_docs
    ordinary_queries = [q for q in range(n_tokens) if q not in sep_positions]
    if ordinary_queries:
      avg_attns["rest_sep"] += attns[:, :, ordinary_queries][:, :, :, sep_positions].sum(-1).mean(-1) / n_docs
  if punct_positions:
    avg_attns["punct"] += attns[:, :, :, punct_positions].sum(-1).mean(-1) / n_docs

Computing relative-position and legacy token stats


### Experiment 2: Attention-sink Candidates (Section 3.2)

The scan excludes self links and the two EOT query rows. Candidate ranking excludes EOT, while EOT statistics remain in the CSV as a baseline. Enrichment is adjusted both for causal-uniform token frequency and for relative-distance buckets. If fewer than two documents were dumped, the helper raises a data-insufficiency error instead of relaxing the thresholds.


In [4]:
sink_stats = compute_token_sink_statistics(
    iter_attention_shards(ATTENTION_DIR),
    eos_token_id=50256,
    min_document_frequency=2,
    min_key_occurrences=10,
    min_average_attention_mass=0.01)

sink_csv = os.path.join(RESULTS_DIR, "token_sink_stats.csv")
sink_fieldnames = list(sink_stats["records"][0])
with open(sink_csv, "w", newline="", encoding="utf-8") as f:
  writer = csv.DictWriter(f, fieldnames=sink_fieldnames)
  writer.writeheader()
  writer.writerows(sink_stats["records"])

print("Top 20 head-token pairs by distance-adjusted enrichment")
for row in sink_stats["top_global"]:
  print("L{:02d}H{:02d} {:>14s} mass={:.4f} enrichment={:.3f}".format(
      row["layer"], row["head"], repr(row["token"]),
      row["average_attention_mass"], row["distance_enrichment"]))

# One top candidate per head. The annotation preserves GPT-2 byte-BPE markers.
top_scores = np.full((n_layers, n_heads), np.nan)
top_labels = np.full((n_layers, n_heads), "", dtype=object)
for (layer, head), rows in sink_stats["top_by_head"].items():
  if rows:
    top_scores[layer - 1, head - 1] = rows[0]["distance_enrichment"]
    top_labels[layer - 1, head - 1] = rows[0]["token"]
plt.figure(figsize=(14, 10))
sns.heatmap(top_scores, annot=top_labels, fmt="", cmap="viridis",
            mask=np.isnan(top_scores),
            xticklabels=np.arange(1, n_heads + 1),
            yticklabels=np.arange(1, n_layers + 1),
            cbar_kws={"label": "Distance-adjusted enrichment"})
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.xlabel("GPT-2 Heads")
plt.ylabel("Layer")
plt.title("Top attention-sink candidate per GPT-2 head (N={})".format(n_docs))
save_figure("experiment2_top_token_heatmap")
plt.show()

# Layer-wise mean attention mass for the highest-ranked unique tokens.
top_token_ids = []
for row in sink_stats["top_global"]:
  if row["token_id"] not in top_token_ids:
    top_token_ids.append(row["token_id"])
  if len(top_token_ids) == 5:
    break
plt.figure(figsize=(8, 5))
for token_id in top_token_ids:
  rows = [r for r in sink_stats["records"]
          if r["token_id"] == token_id and not r["is_eos"]]
  label = rows[0]["token"]
  by_layer = [np.mean([r["average_attention_mass"] for r in rows
                       if r["layer"] == layer])
              for layer in range(1, n_layers + 1)]
  plt.plot(np.arange(1, n_layers + 1), by_layer, marker="o", label=repr(label))
plt.xlabel("Layer")
plt.ylabel("Mean attention mass per ordinary query")
plt.title("Top attention-sink candidates for GPT-2 Heads (N={})".format(n_docs))
if top_token_ids:
  plt.legend(title="GPT-2 token", bbox_to_anchor=(1.02, 1), loc="upper left")
save_figure("experiment2_candidate_layer_attention")
plt.show()

# Frequency diagnostic. One point is one eligible head-token pair.
eligible = [r for r in sink_stats["records"] if r["is_candidate"]]
plt.figure(figsize=(7, 5))
plt.scatter([r["key_occurrences"] for r in eligible],
            [r["distance_enrichment"] for r in eligible],
            c=[r["layer"] for r in eligible], cmap="viridis", s=12, alpha=0.65)
plt.xscale("log")
plt.xlabel("Token key occurrences (log scale)")
plt.ylabel("Distance-adjusted enrichment")
plt.title("Token frequency vs. enrichment for GPT-2 heads (N={})".format(n_docs))
if eligible:
  plt.colorbar(label="Layer")
save_figure("experiment2_frequency_enrichment")
plt.show()

# Mean causal attention per visible link in each distance bucket.
distance_diagnostic = sink_stats["distance_mean"].mean(axis=(0, 1))
plt.figure(figsize=(7, 4))
plt.bar(sink_stats["distance_labels"], distance_diagnostic, color="steelblue")
plt.xlabel("Relative key distance")
plt.ylabel("Mean attention per visible link")
plt.title("Distance baseline diagnostic for GPT-2 heads (N={})".format(n_docs))
save_figure("experiment2_distance_diagnostic")
plt.show()


Top 20 head-token pairs by distance-adjusted enrichment
L03H07            '.' mass=0.1347 enrichment=5.201
L11H04           'ĊĊ' mass=0.0203 enrichment=4.571
L03H07            'Ċ' mass=0.0710 enrichment=4.271
L03H07           'ĊĊ' mass=0.0191 enrichment=3.964
L11H05           'ĊĊ' mass=0.0160 enrichment=3.556
L12H01           'ĊĊ' mass=0.0156 enrichment=3.417
L05H01          'Ġby' mass=0.0135 enrichment=3.056
L02H02            '.' mass=0.0746 enrichment=2.947
L02H12            '.' mass=0.0294 enrichment=2.935
L02H12            'Ċ' mass=0.0181 enrichment=2.840
L03H07            ')' mass=0.0148 enrichment=2.801
L11H05           'Ġ"' mass=0.0104 enrichment=2.789
L03H04         'ĠThe' mass=0.0101 enrichment=2.676
L03H06           'Ġ(' mass=0.0191 enrichment=2.623
L04H02            'Ċ' mass=0.0442 enrichment=2.593
L01H09           'Ġ(' mass=0.0195 enrichment=2.556
L05H01        'Ġwith' mass=0.0132 enrichment=2.540
L04H02           'ĊĊ' mass=0.0116 enrichment=2.538
L03H01          'Ġin' mass

/tmp/ipykernel_34050/1417249802.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_34050/1417249802.py:64: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_34050/1417249802.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_34050/1417249802.py:89: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Experiment 3: Causal Attention Entropy (Section 3.3)
Raw entropy is reported alongside entropy normalized by the size of each query's causal support. Position 0 and the two EOT query rows are excluded from the primary statistic.

In [5]:
entropy_stats = compute_entropy_statistics(
    iter_attention_shards(ATTENTION_DIR))
entropies = entropy_stats["raw"]  # retained for Section 6 compatibility
normalized_entropies = entropy_stats["normalized"]
entropies_trailing_eos = entropy_stats["trailing_eos_raw"]
uniform_attn_entropy = entropy_stats["uniform_raw"]

entropy_csv = os.path.join(RESULTS_DIR, "entropy_by_head.csv")
with open(entropy_csv, "w", newline="", encoding="utf-8") as f:
  fieldnames = ["layer", "head", "raw_entropy",
                "normalized_entropy", "trailing_eos_raw",
                "trailing_eos_normalized", "document_count",
                "query_count"]
  writer = csv.DictWriter(f, fieldnames=fieldnames)
  writer.writeheader()
  for layer in range(n_layers):
    for head in range(n_heads):
      writer.writerow({
          "layer": layer + 1, "head": head + 1,
          "raw_entropy": entropies[layer, head],
          "normalized_entropy": normalized_entropies[layer, head],
          "trailing_eos_raw": entropies_trailing_eos[layer, head],
          "trailing_eos_normalized": entropy_stats["trailing_eos_normalized"][layer, head],
          "document_count": entropy_stats["n_documents"],
          "query_count": entropy_stats["query_count"]})
np.savez(
    os.path.join(RESULTS_DIR, "entropy_by_position.npz"),
    normalized=entropy_stats["position_normalized"],
    document_count=entropy_stats["position_document_count"],
    query_count=entropy_stats["position_query_count"],
    labels=np.asarray(entropy_stats["position_labels"]))

### Plotting Utilities

In [6]:
# Pretty colors
BLACK = "#000000"
GREEN = "#009E73"
SEA = "#56B4E9"
BLUE = "#0072B2"
PURPLE = "#CC79A7"
GREY = "#8C8C8C"
RED = "#D55E00"
ORANGE = "#E69F00"

In [7]:
def get_data_points(head_data):
  xs, ys, avgs = [], [], []
  for layer in range(head_data.shape[0]):
    for head in range(head_data.shape[1]):
      ys.append(head_data[layer, head])
      xs.append(1 + layer)
    avgs.append(head_data[layer].mean())
  return xs, ys, avgs

### Examples of Attention Head Behavior (Figure 1)

In [8]:
width = 3
example_sep = 3
word_height = 1
pad = 0.1


def plot_attn(example, heads):
  """Plots attention maps for the given example and attention heads."""

  for ei, (layer, head) in enumerate(heads):
    yoffset = 1
    xoffset = ei * width * example_sep

    attn = example["attns"][layer][head][-15:, -15:]
    attn = np.array(attn)
    attn /= attn.sum(axis=-1, keepdims=True)
    words = example["tokens"][-15:]
    words[0] = "..."
    n_words = len(words)

    for position, word in enumerate(words):
      plt.text(xoffset + 0, yoffset - position * word_height, word,
               ha="right", va="center")
      plt.text(xoffset + width, yoffset - position * word_height, word,
               ha="left", va="center")
    for i in range(1, n_words):
      for j in range(1, n_words):
        plt.plot([xoffset + pad, xoffset + width - pad],
                 [yoffset - word_height * i, yoffset - word_height * j],
                 color="blue", linewidth=1, alpha=attn[i, j])

plt.figure(figsize=(12, 4))
plt.axis("off")
plot_attn(load_example(min(581, n_docs - 1)),
          [(0, 0), (2, 0), (7, 6), (10, 5)])
plt.title("GPT-2 Heads examples (N={})".format(n_docs))
save_figure("attention_head_examples")
plt.show()

/tmp/ipykernel_34050/2794237115.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Avg. Attention Plots (Figure 2, Sections 3.1 and 3.2)

In [9]:
def add_line(key, ax, color, label, plot_avgs=True):
  xs, ys, avgs = get_data_points(avg_attns[key])
  ax.scatter(xs, ys, s=12, label=label, color=color)
  if plot_avgs:
    ax.plot(1 + np.arange(len(avgs)), avgs, color=color)
  ax.legend(loc="best")
  ax.set_xlabel("Layer")
  ax.set_ylabel("Avg. Attention")


plt.figure(figsize=(5, 10))
ax = plt.subplot(3, 1, 1)
for key, color, label in [
    ("cls", RED, "leading EOT"),
    ("sep", BLUE, "trailing EOT"),
    ("punct", PURPLE, ". or ,"),
]:
  add_line(key, ax, color, label)

ax = plt.subplot(3, 1, 2)
for key, color, label in [
    ("rest_sep", BLUE, "other -> trailing EOT"),
    ("sep_sep", GREEN, "trailing EOT -> EOT"),
]:
  add_line(key, ax, color, label)

ax = plt.subplot(3, 1, 3)
for key, color, label in [
    ("left", RED, "next token"),
    ("right", BLUE, "prev token"),
    ("self", PURPLE, "current token"),
]:
  add_line(key, ax, color, label, plot_avgs=False)

plt.suptitle("GPT-2 Heads average attention (N={})".format(n_docs))
plt.tight_layout()
save_figure("experiment1_average_attention")
plt.show()

/tmp/ipykernel_34050/3278241102.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Entropy Plots (Section 3.3, Figure 4)

In [10]:
xs_raw, raw_points, raw_layer_means = get_data_points(entropies)
xs_norm, norm_points, norm_layer_means = get_data_points(normalized_entropies)

fig, axes = plt.subplots(2, 1, figsize=(7, 8), sharex=True)
axes[0].scatter(xs_raw, raw_points, c=BLUE, s=12, label="GPT-2 Heads")
axes[0].plot(np.arange(1, n_layers + 1), raw_layer_means, c=BLUE,
             label="Layer mean")
axes[0].axhline(uniform_attn_entropy, color=BLACK, linestyle="--",
                label="Mean causal-uniform reference")
axes[0].set_ylabel("Raw entropy (nats)")
axes[0].legend(loc="best")

axes[1].scatter(xs_norm, norm_points, c=PURPLE, s=12, label="GPT-2 Heads")
axes[1].plot(np.arange(1, n_layers + 1), norm_layer_means, c=PURPLE,
             label="Layer mean")
axes[1].axhline(1.0, color=BLACK, linestyle="--", label="Causal uniform")
axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Causal-normalized entropy")
axes[1].legend(loc="best")
fig.suptitle("GPT-2 head entropy (N={})".format(n_docs))
fig.tight_layout()
save_figure("experiment3_entropy_by_head")
plt.show()

position_heatmap = np.nanmean(entropy_stats["position_normalized"], axis=1)
plt.figure(figsize=(9, 6))
sns.heatmap(position_heatmap, cmap="mako", vmin=0, vmax=1,
            xticklabels=entropy_stats["position_labels"],
            yticklabels=np.arange(1, n_layers + 1),
            cbar_kws={"label": "Mean causal-normalized entropy"})
plt.xticks(rotation=0)
plt.yticks(rotation=0)
plt.xlabel("Query-position bucket")
plt.ylabel("Layer")
plt.title("GPT-2 entropy by layer and query position (N={})".format(n_docs))
save_figure("experiment3_entropy_by_position")
plt.show()


/tmp/ipykernel_34050/1106748380.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/tmp/ipykernel_34050/1106748380.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
